# LeetCode 270 — Closest Binary Search Tree Value bằng Python

**Link đề:** https://leetcode.com/problems/closest-binary-search-tree-value/
**Độ khó:** Easy · **Chủ đề:** Binary Search, Tree, Depth-First Search, Binary Search Tree, Binary Tree

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def closestValue(self, root: Optional[TreeNode], target: float) -> int:
        ...
```

## Mục tiêu học
1. Hiểu cây nhị phân, `TreeNode`, và **tính chất BST**: trái < gốc < phải.
2. Tự dựng cây từ list kiểu LeetCode (`[4,2,5,1,3]`) để chạy notebook độc lập.
3. Viết baseline duyệt **toàn bộ** cây, rồi cách **inorder + binary search**.
4. Nắm lời giải chính: **đi một đường từ gốc xuống** như binary search, `O(h)` thời gian, `O(1)` bộ nhớ.
5. Xử lý **hòa khoảng cách** (chọn giá trị nhỏ hơn) và so sánh 4 cách.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho `root` của một **cây nhị phân tìm kiếm** (BST) và số thực `target`. Trả về **giá trị trong cây gần `target` nhất**. Nếu có nhiều đáp án cách đều, trả về **giá trị nhỏ nhất**.

**Ví dụ 1:** `root = [4,2,5,1,3]`, `target = 3.714286` → `4`

```text
        4
       / \
      2   5
     / \
    1   3
|4 - 3.714286| = 0.285714  <  |3 - 3.714286| = 0.714286
```

**Ví dụ 2:** `root = [1]`, `target = 4.428571` → `1`

**Constraints (theo hiểu biết về đề):** số nút trong `[1, 10^4]`; `0 <= Node.val <= 10^9`; `-10^9 <= target <= 10^9`.

Ghi chú: phiên bản cũ của đề nói "đảm bảo chỉ có một đáp án"; phiên bản hiện tại thêm luật **hòa → chọn nhỏ hơn**. Notebook xử lý theo luật mới (vẫn đúng với luật cũ).

In [1]:
target = 3.714286
for v in [1, 2, 3, 4, 5]:
    print(f"v = {v}: |v - target| = {abs(v - target):.6f}")

v = 1: |v - target| = 2.714286
v = 2: |v - target| = 1.714286
v = 3: |v - target| = 0.714286
v = 4: |v - target| = 0.285714
v = 5: |v - target| = 1.285714


## 2. Nền tảng: `TreeNode`, dựng cây, tính chất BST

### 2.1. Class `TreeNode`

Mỗi nút có `val` (giá trị), `left` (con trái), `right` (con phải). `None` nghĩa là không có con. LeetCode đã định nghĩa sẵn class này; trong notebook ta tự viết để chạy độc lập.

In [2]:
from typing import Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

leaf = TreeNode(3)
root_small = TreeNode(2, TreeNode(1), leaf)
print(root_small, root_small.left, root_small.right, root_small.right.left)

TreeNode(2) TreeNode(1) TreeNode(3) None


### 2.2. Dựng cây từ list kiểu LeetCode (level order)

LeetCode ghi cây theo **từng tầng**, trái sang phải, `None` cho chỗ trống: `[4,2,5,1,3]`. Ta dùng hàng đợi `deque` (hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu, đều `O(1)`) để gắn con cho từng nút theo đúng thứ tự.

In [3]:
from collections import deque

def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def print_tree(node, prefix="", is_left=None):
    # In cây nằm ngang: con phải ở trên, con trái ở dưới
    if node is None:
        return
    print_tree(node.right, prefix + ("│   " if is_left else "    "), False)
    print(prefix + ("└── " if is_left else "┌── " if is_left is False else "") + str(node.val))
    print_tree(node.left, prefix + ("    " if is_left else "│   " if is_left is False else "    "), True)

root1 = build_tree([4, 2, 5, 1, 3])
print_tree(root1)

    ┌── 5
4
    │   ┌── 3
    └── 2
        └── 1


Đọc hình: gốc `4` ở giữa; nhánh `┌──` phía trên là con **phải** (`5`), nhánh `└──` phía dưới là con **trái** (`2`).

### 2.3. Tính chất BST

Với **mọi** nút: mọi giá trị ở cây con trái `<` giá trị nút `<` mọi giá trị ở cây con phải.

Hệ quả cực kỳ quan trọng:
- **Duyệt inorder** (trái → gốc → phải) cho dãy **tăng dần**.
- Tìm một giá trị chỉ cần đi **một đường** từ gốc: nhỏ hơn thì rẽ trái, lớn hơn thì rẽ phải (giống binary search trên mảng đã sắp xếp).

In [4]:
def inorder(node, out):
    if node is None:          # điều kiện dừng của đệ quy
        return out
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)
    return out

print("inorder:", inorder(root1, []))

inorder: [1, 2, 3, 4, 5]


### 2.4. `abs`, so sánh tuple và `min` với `key`

- `abs(x)` → giá trị tuyệt đối, dùng đo khoảng cách `abs(v - target)`.
- Python so sánh **tuple** theo thứ tự từng phần tử: `(0.5, 3) < (0.5, 4)` vì phần tử đầu bằng nhau, xét tiếp phần tử thứ hai.
- `min(values, key=f)` chọn phần tử có `f(phần tử)` nhỏ nhất. Dùng `key=lambda v: (abs(v - target), v)` để **vừa** ưu tiên gần nhất **vừa** hòa thì chọn nhỏ hơn.
- `lambda v: ...` là hàm vô danh một dòng.

In [5]:
print((0.5, 3) < (0.5, 4), (0.4, 9) < (0.5, 1))
vals = [3, 4]
target = 3.5
print("min chỉ theo khoảng cách:", min(vals, key=lambda v: abs(v - target)))
print("min theo (khoảng cách, giá trị):", min(vals, key=lambda v: (abs(v - target), v)))
print("min [4, 3] chỉ theo khoảng cách:", min([4, 3], key=lambda v: abs(v - target)), "<- phụ thuộc thứ tự!")

True True
min chỉ theo khoảng cách: 3
min theo (khoảng cách, giá trị): 3
min [4, 3] chỉ theo khoảng cách: 4 <- phụ thuộc thứ tự!


## 3. Cách 1 — Baseline: duyệt **toàn bộ** cây

Bỏ qua tính chất BST, coi như cây thường: thăm mọi nút, giữ giá trị tốt nhất. Đệ quy DFS (Depth-First Search — đi sâu trước).

In [6]:
def closest_all_nodes(root: Optional[TreeNode], target: float) -> int:
    best = root.val
    def dfs(node):
        nonlocal best                      # cho phép gán lại biến best của hàm ngoài
        if node is None:
            return
        if (abs(node.val - target), node.val) < (abs(best - target), best):
            best = node.val
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return best

print(closest_all_nodes(root1, 3.714286), closest_all_nodes(build_tree([1]), 4.428571))

4 1


**`nonlocal`:** hàm con `dfs` muốn **gán lại** `best` của hàm cha. Không có `nonlocal`, Python coi `best` trong `dfs` là biến cục bộ mới → lỗi `UnboundLocalError`.

**Chi phí:** `O(n)` thời gian, `O(h)` bộ nhớ ngăn xếp đệ quy (`h` = chiều cao cây). Đúng nhưng **lãng phí**: không tận dụng BST.

## 4. Cách 2 — Inorder thành mảng tăng dần + binary search (`bisect`)

Inorder cho mảng đã sắp xếp. Trên mảng sắp xếp, giá trị gần `target` nhất là **một trong hai** phần tử kề vị trí chèn `target`.

`bisect.bisect_left(a, x)` trả về vị trí `i` đầu tiên mà `a[i] >= x` (chỗ có thể chèn `x` để mảng vẫn tăng).

In [7]:
import bisect
a = [1, 2, 3, 4, 5]
for x in [3.714286, 0.2, 9, 3]:
    print(f"bisect_left({a}, {x}) = {bisect.bisect_left(a, x)}")

bisect_left([1, 2, 3, 4, 5], 3.714286) = 3
bisect_left([1, 2, 3, 4, 5], 0.2) = 0
bisect_left([1, 2, 3, 4, 5], 9) = 5
bisect_left([1, 2, 3, 4, 5], 3) = 2


In [8]:
def closest_inorder_bisect(root: Optional[TreeNode], target: float) -> int:
    a = inorder(root, [])
    i = bisect.bisect_left(a, target)
    candidates = []
    if i > 0:
        candidates.append(a[i - 1])        # phần tử lớn nhất < target
    if i < len(a):
        candidates.append(a[i])            # phần tử nhỏ nhất >= target
    return min(candidates, key=lambda v: (abs(v - target), v))

print(closest_inorder_bisect(root1, 3.714286), closest_inorder_bisect(build_tree([1]), 4.428571))

4 1


**Chi phí:** inorder vẫn `O(n)` thời gian và `O(n)` bộ nhớ cho mảng. Binary search chỉ `O(log n)`, nhưng bước tạo mảng đã tốn `O(n)`. Nếu phải trả lời **nhiều** truy vấn trên cùng một cây, cách này lại rất tốt (tạo mảng một lần).

## 5. Nút thắt và insight — đi **một đường** như binary search

Câu hỏi: có cần thăm cả cây không?

Đứng tại nút `node`:
- Nếu `target < node.val`: mọi nút ở **cây con phải** đều `> node.val > target` → chúng **xa hơn** `node` → bỏ cả nhánh phải, chỉ đi trái.
- Nếu `target > node.val`: tương tự, bỏ nhánh trái, đi phải.
- Nếu bằng: khoảng cách 0, không thể tốt hơn.

Vậy ta chỉ đi **một đường từ gốc xuống lá**, cập nhật `best` trên đường đi. Số nút thăm = chiều cao `h` (cây cân bằng `h ≈ log n`).

Kiểm chứng ý "nhánh bị bỏ luôn xa hơn" trên ví dụ:

In [9]:
target = 3.714286
node = root1                          # 4
print("Tại 4: target < 4 -> nhánh phải chỉ có", inorder(node.right, []),
      "đều xa hơn 4:", all(abs(v - target) >= abs(4 - target) for v in inorder(node.right, [])))
node = root1.left                     # 2
print("Tại 2: target > 2 -> nhánh trái chỉ có", inorder(node.left, []),
      "đều xa hơn 2:", all(abs(v - target) >= abs(2 - target) for v in inorder(node.left, [])))

Tại 4: target < 4 -> nhánh phải chỉ có [5] đều xa hơn 4: True
Tại 2: target > 2 -> nhánh trái chỉ có [1] đều xa hơn 2: True


## 6. Cách 3 — Lời giải chính: đi xuống BST, giảng từng đoạn

**Đoạn 1 — khởi tạo:** `best = root.val` (cây có ít nhất một nút).

**Đoạn 2 — vòng lặp:** `node = root`, `while node:` (tức `node is not None`).

**Đoạn 3 — cập nhật best** (có luật hòa):
```python
d, db = abs(node.val - target), abs(best - target)
if d < db or (d == db and node.val < best):
    best = node.val
```

**Đoạn 4 — rẽ hướng:**
```python
node = node.left if target < node.val else node.right
```
(`a if điều_kiện else b` là **biểu thức điều kiện** — chọn `a` khi đúng, `b` khi sai.)

**Đoạn 5:** `return best`.

In [10]:
def closest_bst_walk(root: Optional[TreeNode], target: float) -> int:
    best = root.val
    node = root
    while node:
        d, db = abs(node.val - target), abs(best - target)
        if d < db or (d == db and node.val < best):
            best = node.val
        node = node.left if target < node.val else node.right
    return best

print(closest_bst_walk(root1, 3.714286), closest_bst_walk(build_tree([1]), 4.428571))

4 1


### 6.1. Phiên bản đệ quy (cùng ý tưởng)

Đệ quy dễ đọc nhưng tốn `O(h)` ngăn xếp. Với cây lệch 10^4 tầng, Python có giới hạn đệ quy mặc định ~1000 → **có thể lỗi `RecursionError`**. Vì vậy bản vòng lặp an toàn hơn.

In [11]:
import sys
def closest_bst_recursive(node, target, best=None):
    if node is None:
        return best
    if best is None or (abs(node.val - target), node.val) < (abs(best - target), best):
        best = node.val
    nxt = node.left if target < node.val else node.right
    return closest_bst_recursive(nxt, target, best)

print(closest_bst_recursive(root1, 3.714286), "| giới hạn đệ quy mặc định:", sys.getrecursionlimit())

4 | giới hạn đệ quy mặc định: 1000


## 7. TRACE từng bước

Cây lớn hơn: `[8,3,10,1,6,None,14,None,None,4,7,13]`, `target = 5.2`.

| Bước | node | `|node - 5.2|` | best trước | best sau | rẽ |
|---:|---:|---:|---:|---:|---|
| 1 | 8 | 2.8 | 8 | 8 | 5.2 < 8 → trái |
| 2 | 3 | 2.2 | 8 | 3 | 5.2 > 3 → phải |
| 3 | 6 | 0.8 | 3 | 6 | 5.2 < 6 → trái |
| 4 | 4 | 1.2 | 6 | 6 | 5.2 > 4 → phải |
| 5 | None | — | — | — | dừng |

Đáp án `6`. Chỉ thăm 4 / 9 nút.

In [12]:
root2 = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
print_tree(root2)

def trace_closest(root, target):
    best = root.val
    node = root
    step = 0
    print(f"target = {target}")
    while node:
        step += 1
        d, db = abs(node.val - target), abs(best - target)
        old = best
        if d < db or (d == db and node.val < best):
            best = node.val
        direction = "trái" if target < node.val else "phải"
        print(f"  bước {step}: node = {node.val:>2} | khoảng cách = {d:.2f} | best {old} -> {best} | rẽ {direction}")
        node = node.left if target < node.val else node.right
    print(f"  gặp None -> dừng, đáp án = {best}")
    return best

trace_closest(root2, 5.2)

        ┌── 14
        │   └── 13
    ┌── 10
8
    │       ┌── 7
    │   ┌── 6
    │   │   └── 4
    └── 3
        └── 1
target = 5.2
  bước 1: node =  8 | khoảng cách = 2.80 | best 8 -> 8 | rẽ trái
  bước 2: node =  3 | khoảng cách = 2.20 | best 8 -> 3 | rẽ phải
  bước 3: node =  6 | khoảng cách = 0.80 | best 3 -> 6 | rẽ trái
  bước 4: node =  4 | khoảng cách = 1.20 | best 6 -> 6 | rẽ phải
  gặp None -> dừng, đáp án = 6


6

Trace một ca **hòa**: `target = 6.5` giữa `6` và `7` → phải chọn `6`.

In [13]:
trace_closest(root2, 6.5)

target = 6.5
  bước 1: node =  8 | khoảng cách = 1.50 | best 8 -> 8 | rẽ trái
  bước 2: node =  3 | khoảng cách = 3.50 | best 8 -> 8 | rẽ phải
  bước 3: node =  6 | khoảng cách = 0.50 | best 8 -> 6 | rẽ phải
  bước 4: node =  7 | khoảng cách = 0.50 | best 6 -> 6 | rẽ trái
  gặp None -> dừng, đáp án = 6


6

Ở bước gặp `7`, khoảng cách `0.5` bằng với `best = 6` nhưng `7 > 6` nên **không** cập nhật — đúng luật hòa.

## 8. Code nộp LeetCode

In [14]:
class Solution:
    def closestValue(self, root: Optional[TreeNode], target: float) -> int:
        best = root.val
        node = root
        while node:
            d, db = abs(node.val - target), abs(best - target)
            if d < db or (d == db and node.val < best):
                best = node.val
            node = node.left if target < node.val else node.right
        return best

sol = Solution()
print(sol.closestValue(build_tree([4, 2, 5, 1, 3]), 3.714286), sol.closestValue(build_tree([1]), 4.428571))

4 1


Khi nộp: LeetCode đã có `TreeNode` và `Optional` (từ `typing`), chỉ dán class `Solution`.

## 9. Vì sao đúng? (invariant)

Gọi `lo` = giá trị lớn nhất `<= target` trong cây (floor), `hi` = giá trị nhỏ nhất `>= target` (ceiling). Giá trị gần nhất **luôn** là `lo` hoặc `hi` (mọi giá trị khác nằm xa hơn về một phía).

**Invariant:** tại mỗi bước, nếu đáp án đúng chưa được đưa vào `best`, thì nó nằm trong cây con có gốc là `node`.
- Ban đầu `node = root`: đúng.
- Rẽ trái khi `target < node.val`: các nút ở nhánh phải đều lớn hơn `node.val`, xa `target` hơn `node` (đã xét) → không thể là đáp án. Tương tự khi rẽ phải.
- Khi `node = None`: không còn ứng viên nào ngoài những gì đã xét → `best` là đáp án.

Luật hòa: khi hai ứng viên cách đều, điều kiện `node.val < best` giữ lại cái nhỏ hơn bất kể thứ tự gặp.

In [15]:
# Kiểm chứng: đường đi luôn chứa floor và ceiling của target
def path_values(root, target):
    out, node = [], root
    while node:
        out.append(node.val)
        node = node.left if target < node.val else node.right
    return out

for t in [5.2, 6.5, 0, 100, 9]:
    vals = inorder(root2, [])
    floor_ = max([v for v in vals if v <= t], default=None)
    ceil_ = min([v for v in vals if v >= t], default=None)
    p = path_values(root2, t)
    print(f"t = {t}: đường đi = {p}, floor = {floor_}, ceil = {ceil_}, đều trên đường? "
          f"{(floor_ is None or floor_ in p) and (ceil_ is None or ceil_ in p)}")

t = 5.2: đường đi = [8, 3, 6, 4], floor = 4, ceil = 6, đều trên đường? True
t = 6.5: đường đi = [8, 3, 6, 7], floor = 6, ceil = 7, đều trên đường? True
t = 0: đường đi = [8, 3, 1], floor = None, ceil = 1, đều trên đường? True
t = 100: đường đi = [8, 10, 14], floor = 14, ceil = None, đều trên đường? True
t = 9: đường đi = [8, 10], floor = 8, ceil = 10, đều trên đường? True


## 10. Độ phức tạp và so sánh

Gọi `n` = số nút, `h` = chiều cao (`log n` nếu cân bằng, tới `n` nếu lệch).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Duyệt toàn bộ (DFS) | `O(n)` | `O(h)` ngăn xếp | không dùng tính chất BST |
| 2. Inorder + `bisect` | `O(n)` (+ `O(log n)` mỗi truy vấn sau) | `O(n)` | tốt khi nhiều truy vấn |
| 3. Đi xuống BST (vòng lặp) | `O(h)` | `O(1)` | **lời giải chính** |
| 3b. Đi xuống BST (đệ quy) | `O(h)` | `O(h)` | có thể chạm giới hạn đệ quy |

In [16]:
# Minh họa h: cây cân bằng vs cây lệch, số nút thăm
def build_balanced(vals):
    if not vals:
        return None
    mid = len(vals) // 2
    return TreeNode(vals[mid], build_balanced(vals[:mid]), build_balanced(vals[mid + 1:]))

def build_skewed(vals):              # chuỗi toàn con phải: 0 -> 1 -> 2 -> ...
    root = None
    for v in reversed(vals):
        root = TreeNode(v, None, root)
    return root

N = 10000
bal, skw = build_balanced(list(range(N))), build_skewed(list(range(N)))
print("số nút thăm, cây cân bằng:", len(path_values(bal, 9876.3)))
print("số nút thăm, cây lệch    :", len(path_values(skw, 9876.3)))
print("đáp án cả hai:", Solution().closestValue(bal, 9876.3), Solution().closestValue(skw, 9876.3))

số nút thăm, cây cân bằng: 13
số nút thăm, cây lệch    : 9878
đáp án cả hai: 9876 9876


## 11. Bộ test

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.closestValue(build_tree([4, 2, 5, 1, 3]), 3.714286), 4)
check("mẫu 2: một nút", sol.closestValue(build_tree([1]), 4.428571), 1)
check("hòa 3.5 giữa 3 và 4 -> nhỏ hơn", sol.closestValue(build_tree([4, 2, 5, 1, 3]), 3.5), 3)
check("hòa 6.5 giữa 6 và 7 -> 6", sol.closestValue(root2, 6.5), 6)
check("target trùng giá trị", sol.closestValue(root2, 10), 10)
check("target nhỏ hơn mọi nút", sol.closestValue(root2, -1e9), 1)
check("target lớn hơn mọi nút", sol.closestValue(root2, 1e9), 14)
check("giá trị lớn 10^9", sol.closestValue(build_tree([10**9]), 0.0), 10**9)

mẫu 1                                  PASS (4)
mẫu 2: một nút                         PASS (1)
hòa 3.5 giữa 3 và 4 -> nhỏ hơn         PASS (3)
hòa 6.5 giữa 6 và 7 -> 6               PASS (6)
target trùng giá trị                   PASS (10)
target nhỏ hơn mọi nút                 PASS (1)
target lớn hơn mọi nút                 PASS (14)
giá trị lớn 10^9                       PASS (1000000000)


In [18]:
import random

def random_bst(rng, size, lo=0, hi=60):
    vals = rng.sample(range(lo, hi), size)
    root = None
    def insert(node, v):
        if node is None:
            return TreeNode(v)
        if v < node.val:
            node.left = insert(node.left, v)
        else:
            node.right = insert(node.right, v)
        return node
    for v in vals:
        root = insert(root, v)
    return root

rng = random.Random(270)
for _ in range(1500):
    r = random_bst(rng, rng.randint(1, 25))
    t = rng.choice([rng.uniform(-5, 65), rng.randint(-5, 65) + 0.5, float(rng.randint(-5, 65))])
    expected = closest_all_nodes(r, t)
    got = {closest_inorder_bisect(r, t), closest_bst_walk(r, t), closest_bst_recursive(r, t), Solution().closestValue(r, t)}
    assert got == {expected}, (inorder(r, []), t, got, expected)
print("4 cách khớp nhau trên 1500 cây ngẫu nhiên (có cả target .5 để thử hòa)")

4 cách khớp nhau trên 1500 cây ngẫu nhiên (có cả target .5 để thử hòa)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dừng ngay khi thấy nút "đủ gần" hoặc chỉ so với con trực tiếp

Có người viết: "nếu `target < node.val` và `node.left` xa hơn `node` thì trả `node`". Sai vì cháu của `node.left` có thể gần hơn.

In [19]:
def wrong_greedy_stop(root, target):
    node = root
    while node:
        nxt = node.left if target < node.val else node.right
        if nxt is None or abs(nxt.val - target) >= abs(node.val - target):
            return node.val                  # SAI: dừng quá sớm
        node = nxt
    return root.val

print("wrong_greedy_stop(root2, 5.2) =", wrong_greedy_stop(root2, 5.2), "| đúng =", Solution().closestValue(root2, 5.2))

wrong_greedy_stop(root2, 5.2) = 6 | đúng = 6


Ở ví dụ này bản sai **vô tình** đúng (mỗi bước con đều gần hơn cha cho tới nút 6). Nhưng "con xa hơn cha" **không** có nghĩa "cả nhánh con xa hơn": cháu có thể gần hơn cả hai. **Sửa:** luôn đi tới `None`, cập nhật `best` dọc đường.

Phản ví dụ: cây `10 → (trái) 2 → (phải) 9`, `target = 8.8`. Tại `10` (cách 1.2), con `2` cách 6.8 nên bản sai dừng và trả `10`; nhưng cháu `9` chỉ cách 0.2.

In [20]:
r = build_tree([10, 2, None, None, 9])   # 10 -> trái 2 -> phải 9
print_tree(r)
print("target 8.8: wrong =", wrong_greedy_stop(r, 8.8), "| đúng =", Solution().closestValue(r, 8.8))

10
    │   ┌── 9
    └── 2
target 8.8: wrong = 10 | đúng = 9


### Lỗi 2: Quên luật hòa (dùng `<` đơn thuần)

In [21]:
def wrong_tie(root, target):
    best, node = root.val, root
    while node:
        if abs(node.val - target) < abs(best - target):     # hòa -> giữ cái gặp trước
            best = node.val
        node = node.left if target < node.val else node.right
    return best

r = build_tree([4, 2, 5, 1, 3])
print("target 4.5: wrong =", wrong_tie(r, 4.5), "| đúng =", Solution().closestValue(r, 4.5))

target 4.5: wrong = 4 | đúng = 4


Ở đây `4` gặp trước nên vô tình đúng; nhưng target `3.5`: gặp `4` trước (khoảng cách 0.5), sau đó `3` cũng 0.5 → bản sai giữ `4`:

In [22]:
print("target 3.5: wrong =", wrong_tie(r, 3.5), "| đúng =", Solution().closestValue(r, 3.5))

target 3.5: wrong = 4 | đúng = 3


### Lỗi 3: Khởi tạo `best = 0` hoặc `float('inf')` rồi trả nhầm

`best = 0`: nếu mọi nút xa hơn `0` (ví dụ cây `[100]`, target `1`) thì trả `0` — giá trị không có trong cây! Luôn khởi tạo bằng `root.val`.

In [23]:
def wrong_init(root, target):
    best, node = 0, root                                  # SAI
    while node:
        if abs(node.val - target) < abs(best - target):
            best = node.val
        node = node.left if target < node.val else node.right
    return best

print("wrong_init([100], 1) =", wrong_init(build_tree([100]), 1), "| đúng =", Solution().closestValue(build_tree([100]), 1))

wrong_init([100], 1) = 0 | đúng = 100


### Lỗi 4: Rẽ ngược hướng (`target > node.val` thì đi trái)

Đi sai hướng sẽ bỏ qua đúng nhánh chứa đáp án. Mẹo nhớ: **target nhỏ → đi về phía giá trị nhỏ = trái**.

## 13. Mẹo nhận dạng pattern

- **BST + "tìm / gần nhất / floor / ceiling"** → đi một đường từ gốc, `O(h)`, như binary search.
- **"Gần nhất" trên dãy đã sắp xếp** → ứng viên chỉ là hai phần tử kề vị trí chèn (`bisect`).
- **"Hòa thì chọn nhỏ hơn"** → so sánh tuple `(khoảng_cách, giá_trị)`.
- Cần **k** giá trị gần nhất (bài 272, Hard) → inorder + cửa sổ trượt / hai stack.

Bài liên quan: 700 Search in a BST, 701 Insert into a BST, 285 Inorder Successor in BST, 272 Closest BST Value II, 35 Search Insert Position.

## 14. Biến thể

### 14.1. Floor và ceiling trong BST

In [24]:
def bst_floor(root, target):
    ans, node = None, root
    while node:
        if node.val <= target:
            ans = node.val           # ứng viên floor, thử tìm lớn hơn ở bên phải
            node = node.right
        else:
            node = node.left
    return ans

def bst_ceiling(root, target):
    ans, node = None, root
    while node:
        if node.val >= target:
            ans = node.val
            node = node.left
        else:
            node = node.right
    return ans

for t in [5.2, 0, 15, 7]:
    print(f"t = {t}: floor = {bst_floor(root2, t)}, ceiling = {bst_ceiling(root2, t)}")

t = 5.2: floor = 4, ceiling = 6
t = 0: floor = None, ceiling = 1
t = 15: floor = 14, ceiling = None
t = 7: floor = 7, ceiling = 7


### 14.2. `k` giá trị gần nhất (ý tưởng bài 272) — bản đơn giản `O(n log n)`

In [25]:
def k_closest_simple(root, target, k):
    vals = inorder(root, [])
    return sorted(vals, key=lambda v: (abs(v - target), v))[:k]

print(k_closest_simple(root2, 5.2, 3))

[6, 4, 7]


### 14.3. Cây nhị phân **thường** (không phải BST)

Không còn tính chất trái < gốc < phải → **phải** duyệt toàn bộ (cách 1). Đây là lý do nhận dạng "BST" trong đề rất quan trọng.

In [26]:
not_bst = build_tree([5, 9, 1, 2, 8])
print_tree(not_bst)
print("duyệt toàn bộ:", closest_all_nodes(not_bst, 1.1), "| đi một đường (sai với cây thường):", closest_bst_walk(not_bst, 1.1))

    ┌── 1
5
    │   ┌── 8
    └── 9
        └── 2
duyệt toàn bộ: 1 | đi một đường (sai với cây thường): 2


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Tính tay:** Với `root2` và `target = 11.9`, liệt kê các nút trên đường đi và đáp án.

In [27]:
# TODO
predict_path = None      # list giá trị các nút được thăm
predict_answer = None
# assert predict_path == path_values(root2, 11.9) and predict_answer == Solution().closestValue(root2, 11.9)

**Bài 2 — Điền chỗ trống** trong bản vòng lặp.

In [28]:
def exercise_closest(root, target):
    best = root.val
    node = root
    while node:
        # TODO 1: cập nhật best (nhớ luật hòa)
        # TODO 2: rẽ trái hay phải?
        pass
    return best

# assert exercise_closest(build_tree([4, 2, 5, 1, 3]), 3.714286) == 4

**Bài 3 — Search in a BST (LeetCode 700):** trả về nút có `val == x` hoặc `None`.

In [29]:
def exercise_search_bst(root, x):
    # TODO
    pass

# assert exercise_search_bst(root2, 6).val == 6 and exercise_search_bst(root2, 5) is None

**Bài 4 — Closest với target nguyên, trả về cả khoảng cách:** `(value, distance)`.

In [30]:
def exercise_closest_with_dist(root, target):
    # TODO
    pass

# assert exercise_closest_with_dist(root2, 12) == (13, 1)

**Bài 5 — Đếm số nút có giá trị trong `[L, R]`** chỉ thăm các nhánh cần thiết (Range Sum of BST, bài 938 – đổi thành đếm).

In [31]:
def exercise_count_range(root, L, R):
    # TODO
    pass

# assert exercise_count_range(root2, 4, 10) == 5      # 4, 6, 7, 8, 10

**Bài 6 — Mini challenge:** `k` giá trị gần nhất trong `O(n)` bằng inorder + **cửa sổ trượt** `deque` độ dài `k`.

In [32]:
def exercise_k_closest_window(root, target, k):
    # TODO: duyệt inorder; nếu window đủ k và phần tử mới gần hơn phần tử đầu thì popleft
    pass

# assert sorted(exercise_k_closest_window(root2, 5.2, 3)) == sorted(k_closest_simple(root2, 5.2, 3))

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | bắt đầu từ 8 | 11.9 > 8 → phải | so tiếp với 10, 14, 13 |
| 2 | xem mục 6 đoạn 3–4 | dùng tuple hoặc `d < db or (d == db and ...)` | `node = node.left if target < node.val else node.right` |
| 3 | giống đi xuống BST | bằng thì trả `node` | hết đường thì `None` |
| 4 | tính `best` như bài chính | trả `(best, abs(best - target))` | |
| 5 | đệ quy, cắt nhánh | `node.val < L` → chỉ cần phải | đếm `1` nếu `L <= val <= R` rồi đi cả hai nhánh |
| 6 | inorder cho dãy tăng | window đầy: so `abs(v - t)` với `abs(window[0] - t)` | nếu không gần hơn thì có thể dừng sớm |

## 17. Đáp án tham khảo

In [33]:
answer_path = [8, 10, 14, 13]
answer_value = 13
assert answer_path == path_values(root2, 11.9) and answer_value == Solution().closestValue(root2, 11.9)
print("Bài 1:", answer_path, "->", answer_value, "(|13-11.9| = 1.1 < |10-11.9| = 1.9)")

def answer_closest(root, target):
    best, node = root.val, root
    while node:
        if (abs(node.val - target), node.val) < (abs(best - target), best):
            best = node.val
        node = node.left if target < node.val else node.right
    return best

def answer_search_bst(root, x):
    node = root
    while node and node.val != x:
        node = node.left if x < node.val else node.right
    return node

def answer_closest_with_dist(root, target):
    v = answer_closest(root, target)
    return (v, abs(v - target))

def answer_count_range(node, L, R):
    if node is None:
        return 0
    if node.val < L:
        return answer_count_range(node.right, L, R)
    if node.val > R:
        return answer_count_range(node.left, L, R)
    return 1 + answer_count_range(node.left, L, R) + answer_count_range(node.right, L, R)

def answer_k_closest_window(root, target, k):
    window = deque()
    stop = False
    def visit(node):
        nonlocal stop
        if node is None or stop:
            return
        visit(node.left)
        if stop:
            return
        v = node.val
        if len(window) < k:
            window.append(v)
        elif abs(v - target) < abs(window[0] - target):
            window.popleft()
            window.append(v)
        else:
            stop = True          # dãy tăng: các giá trị sau còn xa hơn
            return
        visit(node.right)
    visit(root)
    return list(window)

assert answer_closest(build_tree([4, 2, 5, 1, 3]), 3.714286) == 4
assert answer_search_bst(root2, 6).val == 6 and answer_search_bst(root2, 5) is None
assert answer_closest_with_dist(root2, 12) == (13, 1)
assert answer_count_range(root2, 4, 10) == 5
assert sorted(answer_k_closest_window(root2, 5.2, 3)) == sorted(k_closest_simple(root2, 5.2, 3))
rng = random.Random(2700)
for _ in range(300):
    r = random_bst(rng, rng.randint(1, 20))
    t = rng.uniform(-5, 65) + 0.01      # tránh hòa để tập k phần tử là duy nhất
    k = rng.randint(1, len(inorder(r, [])))
    assert sorted(answer_k_closest_window(r, t, k)) == sorted(k_closest_simple(r, t, k))
print("Tất cả đáp án PASS")

Bài 1: [8, 10, 14, 13] -> 13 (|13-11.9| = 1.1 < |10-11.9| = 1.9)
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def closestValue(self, root: Optional[TreeNode], target: float) -> int:
        best, node = root.val, root
        while node:
            if (abs(node.val - target), node.val) < (abs(best - target), best):
                best = node.val
            node = node.left if target < node.val else node.right
        return best
```

**Câu thần chú:** *"BST = binary search trên cây: cập nhật best, nhỏ thì trái, lớn thì phải, tới None thì dừng."*

**Checklist:**
- [ ] `best` khởi tạo bằng `root.val`, không phải 0.
- [ ] Đi tới khi `node is None`, không dừng sớm.
- [ ] Hướng rẽ: `target < node.val` → trái.
- [ ] Luật hòa: chọn giá trị nhỏ hơn.
- [ ] Ưu tiên vòng lặp (tránh giới hạn đệ quy với cây lệch).
- [ ] Tên method `closestValue`, trả `int`.